# Notebook for testing a local fork of the following tax library:
https://github.com/blackace007-cmd/tax-calculator

Not to be confused with the other package by the same name.


In [1]:
#Need to add to the path, since tax-calculator is not a valid package name directory.
import sys
sys.path.append('/home/filip/financial/tax-calculator')

In [2]:
import taxcalc

In [3]:
taxcalc.__version__

'1.1.0'

# Testing the Policy data construct.

A `Policy` represents **one tax** (`income`, `niit`) for one jurisdiction and
year. Parameters are resolved from a versioned catalog, so the same rules can be
reused across tax years.


In [4]:
from taxcalc import (
    FilingStatus,
    Person,
    TaxUnit,
    Income,
    build_policy,
    compute_tax,
    compute_return,
    param,
)


In [5]:
taxpayer = TaxUnit(
    filing_status=FilingStatus.MARRIED_JOINT,
    primary=Person(age=45),
    spouse=Person(age=43),
    dependents=[
        Person(age=5, relationship="child", full_time_student=True),
        Person(age=12, relationship="child", full_time_student=True)
    ],
    income=Income(
        wages_w2=400000,
        taxable_interest=5000,
        ordinary_dividends=55000,
        qualified_dividends=45000,
        long_term_capital_gain_loss=100000,
        other_sched_1=20000
    )


)



## Build policies from the parameter catalog

`build_policy` resolves every parameter whose effective date range covers the
tax year. `yearly(...)` entries apply to one year; `since(...)` entries stay in
force until amended.


In [6]:
fed_2023 = build_policy("federal", "income", 2023)
fed_2024 = build_policy("federal", "income", 2024)

# Resolved values are plain data, so they are easy to inspect.
print("2023 brackets (MFJ):", param(fed_2023, "brackets", "married_joint"))
print("2024 brackets (MFJ):", param(fed_2024, "brackets", "married_joint"))
print(
    "2024 standard deduction (MFJ):",
    param(fed_2024, "standard_deduction", "married_joint"),
)


2023 brackets (MFJ): [(0.1, 22000), (0.12, 89450), (0.22, 190750)]
2024 brackets (MFJ): [(0.1, 23200), (0.12, 94300), (0.22, 201050), (0.24, 383900), (0.32, 487450), (0.35, 731200), (0.37, inf)]
2024 standard deduction (MFJ): 29200


## Same rules, different tax year

Both policies use the same 2023-2025 rule set; only the parameters differ.


In [7]:
result_2023 = compute_tax(taxpayer, fed_2023)
result_2024 = compute_tax(taxpayer, fed_2024)

print("2023:", result_2023)
print("2024:", result_2024)


2023: TaxResult(agi=580000.0, taxable_income=552300.0, pre_credit_tax=112121.0, total_credits=4000, net_tax=108121.0)
2024: TaxResult(agi=580000.0, taxable_income=550800.0, pre_credit_tax=133529.5, total_credits=4000, net_tax=129529.5)


## What-if: override a resolved parameter

Overrides replace a resolved value, which is handy for testing policy changes
without editing the catalog.


In [8]:
hypothetical = build_policy(
    "federal", "income", 2024,
    overrides={"child_credit_amount": {"default": [4000]}},
)
print("2024 with a $4,000 child credit:", compute_tax(taxpayer, hypothetical))


2024 with a $4,000 child credit: TaxResult(agi=580000.0, taxable_income=550800.0, pre_credit_tax=133529.5, total_credits=8000, net_tax=125529.5)


## Multiple taxes: income + NIIT

`compute_return` calculates AGI once, runs each tax policy, and combines the
results.


In [9]:
niit_2024 = build_policy("federal", "niit", 2024)
combined = compute_return(taxpayer, {"income": fed_2024, "niit": niit_2024})

print("AGI:", combined.agi)
print("parts:", combined.parts)
print("total tax:", combined.total_tax)


AGI: 580000.0
parts: {'income': 129529.5, 'niit': 6840.0}
total tax: 136369.5


## Effective-date gating

There is no fallback. Asking for a year with no effective parameter or rule
set raises an error that lists what is available.


In [10]:
from taxcalc import NoEffectiveVersionError

try:
    build_policy("federal", "income", 2026)
except NoEffectiveVersionError as e:
    print(type(e).__name__, "->", e)


NoEffectiveVersionError -> federal/income/brackets: no version is effective for tax year 2026. Available ranges: 2023-01-01..2023-12-31, 2024-01-01..2024-12-31


## Catalog inspection

Each `yearly(...)` entry covers one year. Supporting 2026 means adding
`yearly(2026, ...)` entries to the catalog.


In [11]:
from taxcalc import PARAMETERS

for version in PARAMETERS[("federal", "income", "standard_deduction")]:
    print(version.effective, "->", version.value["married_joint"])


(datetime.date(2023, 1, 1), datetime.date(2023, 12, 31)) -> [27700]
(datetime.date(2024, 1, 1), datetime.date(2024, 12, 31)) -> [29200]
